# RigSmith PCs: a ReAct build-advisor agent with configurable personas
**EECE 503P/798S: Agentic Systems** · C4 – Agent Personas & Thinking Mechanisms · **Student:** Adam Hijazi

This notebook extends my C3 business chatbot (RigSmith PCs, a fictional custom-PC studio in Hamra, Beirut) into a **ReAct agent** built with **LangGraph**. The Thought → Action → Observation loop is implemented by hand as a state graph; no prebuilt executor is used.

| Section | Content | Rubric item |
|---|---|---|
| 1 | Use-case summary | Use-case write-up (10) |
| 2 | Tools | ReAct logic (30) |
| 3 | Instructions block (with a worked example) | Clear instructions (10) |
| 4 | Three personas: prompt traits + LLM configuration | Persona design (20) |
| 5 | The ReAct loop as a LangGraph graph | ReAct logic (30) |
| 6 | Experiments: personas, prompt engineering, configurations | Experimentation (15) |
| 7 | Results and reflection | Reflection (15) |
| 8 | Gradio chat app with a persona selector | Working assistant |

**Run it:** add your Gemini API key as a Colab secret named `GEMINI_API_KEY` (🔑 icon in the left sidebar), then `Runtime → Run all`. No GPU is needed. The experiments make roughly 100 API calls, so they are paced to respect the free tier's per-minute limit and **saved as they go**: if the run stops, re-running the cell continues where it left off.

In [1]:
!pip -q install google-genai langgraph "gradio>=5,<6" pypdf pandas

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
hf-gradio 0.4.1 requires gradio-client<3.0,>=2.0, but you have gradio-client 1.14.0 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 2.8.0 which is incompatible.
google-adk 2.7.1 requires starlette<2,>=1.3.1, but you have starlette 0.52.1 which is incompatible.
python-fasthtml 0.14.18 requires starlette>=1.0.1, but you have starlette 0.52.1 which is incompatible.


In [2]:
import os, re, json, time, math, datetime
from typing import TypedDict, Optional, Any
import pandas as pd
from google import genai
from google.genai import types

API_KEY = os.getenv("GEMINI_API_KEY")
if not API_KEY:
    try:
        from google.colab import userdata
        API_KEY = userdata.get("GEMINI_API_KEY")
    except Exception:
        from getpass import getpass
        API_KEY = getpass("Paste your Gemini API key: ")
client = genai.Client(api_key=API_KEY)

# Business documents from C3 (upload them in Colab if the me/ folder is missing)
if not os.path.exists("me/business_summary.txt"):
    try:
        from google.colab import files
        os.makedirs("me", exist_ok=True)
        print("Upload about_business.pdf and business_summary.txt from the C3 project")
        for name, data in files.upload().items():
            open(os.path.join("me", name), "wb").write(data)
    except ImportError:
        raise FileNotFoundError("Put about_business.pdf and business_summary.txt in a folder called me/")
print("Ready. Business files:", sorted(os.listdir("me")))

Upload about_business.pdf and business_summary.txt from the C3 project


Saving about_business.pdf to about_business.pdf
Saving business_summary.txt to business_summary.txt
Ready. Business files: ['about_business.pdf', 'business_summary.txt']


## 1. Use case

| | |
|---|---|
| **Business** | RigSmith PCs, a fictional custom-PC building studio in Hamra, Beirut (reused from C3) |
| **Agent goal** | Help customers **plan a custom PC**: pick parts that fit their budget and use case, check that the parts are compatible, give an itemized price, estimate build and delivery time, answer policy questions, and save interested customers as leads. |
| **Target audience** | Gamers, university students, and content creators in Lebanon, from first-time buyers to hardware enthusiasts. |
| **Tools** | `search_parts`, `price_build`, `check_compatibility`, `estimate_delivery`, `lookup_policy`, `record_lead` (Section 2) |
| **Constraints** | Prices come **only** from the RigSmith parts catalog (a fictional catalog; the prices are not real market prices), and policies only from the company documents. The agent never invents prices, never guarantees stock, and never asks for payment details. It may take at most 8 tool steps per question. Delivery prices exist only for Lebanon. |
| **Limitations** | The catalog is small (about 30 parts), compatibility checks cover the most common problems (CPU socket, RAM type, PSU wattage, case size) rather than every detail, and the model is a free-tier Gemini Flash model, so rate limits apply. |

**Why ReAct?** Planning a PC needs several dependent steps: search for parts, then price them, then check compatibility, and sometimes go back and swap a part. A single prompt would have to guess prices and compatibility. ReAct lets the model *reason*, *call a tool*, *read the real result*, and continue, so every number in the final answer comes from a tool.

## 2. Tools

Six plain Python functions. All data (the catalog, delivery zones, documents) is local, so the tools are deterministic and the experiments are repeatable. Each tool returns a short text **observation** that is fed back to the model.

In [3]:
from pypdf import PdfReader

# ---------------- RigSmith parts catalog (fictional prices in USD) ----------------
CATALOG = [
    # id, category, name, price, specs
    ("cpu_r5_7600",   "cpu", "AMD Ryzen 5 7600 (6 cores)",       189, {"socket": "AM5", "watts": 65}),
    ("cpu_r5_9600x",  "cpu", "AMD Ryzen 5 9600X (6 cores)",      229, {"socket": "AM5", "watts": 65}),
    ("cpu_r7_9700x",  "cpu", "AMD Ryzen 7 9700X (8 cores)",      329, {"socket": "AM5", "watts": 65}),
    ("cpu_r7_7800x3d","cpu", "AMD Ryzen 7 7800X3D (8 cores, gaming)", 399, {"socket": "AM5", "watts": 120}),
    ("cpu_r5_5600",   "cpu", "AMD Ryzen 5 5600 (6 cores)",       109, {"socket": "AM4", "watts": 65}),
    ("cpu_i5_12400f", "cpu", "Intel Core i5-12400F (6 cores)",   119, {"socket": "LGA1700", "watts": 65}),
    ("cpu_i5_14600k", "cpu", "Intel Core i5-14600K (14 cores)",  279, {"socket": "LGA1700", "watts": 125}),
    ("mb_b650",  "motherboard", "B650 ATX motherboard (AM5, DDR5)",     159, {"socket": "AM5", "ram": "DDR5", "form": "ATX"}),
    ("mb_b650m", "motherboard", "B650M micro-ATX motherboard (AM5, DDR5)", 129, {"socket": "AM5", "ram": "DDR5", "form": "mATX"}),
    ("mb_b550",  "motherboard", "B550 ATX motherboard (AM4, DDR4)",     109, {"socket": "AM4", "ram": "DDR4", "form": "ATX"}),
    ("mb_b760",  "motherboard", "B760 ATX motherboard (LGA1700, DDR5)", 149, {"socket": "LGA1700", "ram": "DDR5", "form": "ATX"}),
    ("gpu_rtx5060",   "gpu", "NVIDIA RTX 5060 8GB (1080p high)",        299, {"watts": 145, "level": "1080p"}),
    ("gpu_rtx5060ti", "gpu", "NVIDIA RTX 5060 Ti 16GB (1440p / editing)", 449, {"watts": 180, "level": "1440p"}),
    ("gpu_rx9070",    "gpu", "AMD RX 9070 16GB (1440p high)",           549, {"watts": 220, "level": "1440p"}),
    ("gpu_rtx5070",   "gpu", "NVIDIA RTX 5070 12GB (1440p high / streaming)", 579, {"watts": 250, "level": "1440p"}),
    ("gpu_rtx5070ti", "gpu", "NVIDIA RTX 5070 Ti 16GB (4K entry)",      799, {"watts": 300, "level": "4K"}),
    ("gpu_rtx5080",   "gpu", "NVIDIA RTX 5080 16GB (4K high)",         1099, {"watts": 360, "level": "4K"}),
    ("ram_16_ddr4", "ram", "16GB DDR4-3200 (2x8)",  39, {"type": "DDR4"}),
    ("ram_32_ddr4", "ram", "32GB DDR4-3600 (2x16)", 69, {"type": "DDR4"}),
    ("ram_32_ddr5", "ram", "32GB DDR5-6000 (2x16)", 99, {"type": "DDR5"}),
    ("ram_64_ddr5", "ram", "64GB DDR5-6000 (2x32)", 189, {"type": "DDR5"}),
    ("ssd_1tb", "storage", "1TB NVMe SSD",  69, {}),
    ("ssd_2tb", "storage", "2TB NVMe SSD", 129, {}),
    ("psu_650", "psu", "650W 80+ Bronze PSU", 69,  {"watts": 650}),
    ("psu_750", "psu", "750W 80+ Gold PSU",   99,  {"watts": 750}),
    ("psu_850", "psu", "850W 80+ Gold PSU",  129,  {"watts": 850}),
    ("case_atx",  "case", "Airflow mid-tower case (fits ATX and mATX)", 79, {"fits": ["ATX", "mATX"]}),
    ("case_matx", "case", "Compact case (fits mATX only)",              65, {"fits": ["mATX"]}),
    ("cooler_air", "cooler", "Dual-tower air cooler", 39, {}),
]
PARTS = {pid: {"id": pid, "category": cat, "name": name, "price": price, **specs} for pid, cat, name, price, specs in CATALOG}
ASSEMBLY_LABOR = 40          # assembly + 48-hour stress test (same as the "assembly" service in the company profile)
TIERS = [("Starter", 650), ("Pro Gamer", 1200), ("Creator Workstation", 1800), ("Beast Mode", 2800)]

def _ids(part_ids):
    if isinstance(part_ids, str):
        part_ids = [p.strip() for p in re.split(r"[,\s]+", part_ids) if p.strip()]
    return list(part_ids)

# ---------------- Tool 1 ----------------
def search_parts(category, max_price: float = None) -> str:
    # category: one category, a list of categories, or "all"
    if isinstance(category, list) or (isinstance(category, str) and ("," in category or category.lower().strip() == "all")):
        cats = sorted({p["category"] for p in PARTS.values()}) if str(category).lower().strip() == "all" else \
               (category if isinstance(category, list) else [c.strip() for c in category.split(",")])
        return "\n\n".join(f"[{c.upper()}]\n" + search_parts(c, max_price) for c in cats)
    cat = category.lower().strip().rstrip("s")
    cat = {"processor": "cpu", "graphics card": "gpu", "graphic": "gpu", "memory": "ram", "ssd": "storage",
           "power supply": "psu", "mobo": "motherboard", "board": "motherboard"}.get(cat, cat)
    items = [p for p in PARTS.values() if p["category"] == cat and (max_price is None or p["price"] <= float(max_price))]
    if not items:
        cats = sorted({p["category"] for p in PARTS.values()})
        return f"No parts found for category '{category}' (max_price={max_price}). Valid categories: {', '.join(cats)}."
    lines = []
    for p in sorted(items, key=lambda x: x["price"]):
        specs = ", ".join(f"{k}={v}" for k, v in p.items() if k not in ("id", "category", "name", "price"))
        lines.append(f"{p['id']}: {p['name']} - ${p['price']}" + (f" ({specs})" if specs else ""))
    return "\n".join(lines)

# ---------------- Tool 2 ----------------
def price_build(part_ids) -> str:
    ids = _ids(part_ids)
    unknown = [i for i in ids if i not in PARTS]
    if unknown:
        return f"Unknown part id(s): {unknown}. Use search_parts to get valid ids."
    lines = [f"- {PARTS[i]['name']}: ${PARTS[i]['price']}" for i in ids]
    subtotal = sum(PARTS[i]["price"] for i in ids)
    total = subtotal + ASSEMBLY_LABOR
    tier = next((name for name, start in reversed(TIERS) if total >= start), "below Starter range")
    missing = [c for c in ("cpu", "motherboard", "ram", "storage", "psu", "case") if c not in {PARTS[i]["category"] for i in ids}]
    note = f"\nMissing categories for a complete PC: {', '.join(missing)}" if missing else ""
    return "\n".join(lines) + f"\nParts subtotal: ${subtotal}\nAssembly + 48h stress test: ${ASSEMBLY_LABOR}\nTOTAL: ${total} (tier: {tier}){note}"

# ---------------- Tool 3 ----------------
def check_compatibility(part_ids) -> str:
    ids = [i for i in _ids(part_ids) if i in PARTS]
    by = {}
    for i in ids:
        by.setdefault(PARTS[i]["category"], PARTS[i])
    issues, ok = [], []
    cpu, mb, ram, psu, case, gpu = (by.get(c) for c in ("cpu", "motherboard", "ram", "psu", "case", "gpu"))
    if cpu and mb:
        (ok if cpu["socket"] == mb["socket"] else issues).append(
            f"CPU socket {cpu['socket']} vs motherboard socket {mb['socket']}")
    if ram and mb:
        (ok if ram["type"] == mb["ram"] else issues).append(f"RAM {ram['type']} vs motherboard {mb['ram']}")
    if mb and case:
        (ok if mb["form"] in case["fits"] else issues).append(f"{mb['form']} motherboard in case that fits {'/'.join(case['fits'])}")
    if psu:
        need = (cpu["watts"] if cpu else 0) + (gpu["watts"] if gpu else 0) + 150
        recommended = int(math.ceil(need / 50) * 50)
        (ok if psu["watts"] >= recommended else issues).append(f"PSU {psu['watts']}W vs recommended {recommended}W")
    if not (ok or issues):
        return "Not enough parts to check. Give at least a CPU and a motherboard, or a PSU with a CPU/GPU."
    verdict = "COMPATIBLE" if not issues else "NOT COMPATIBLE"
    return verdict + "".join(f"\n  OK: {o}" for o in ok) + "".join(f"\n  PROBLEM: {p}" for p in issues)

# ---------------- Tool 4 ----------------
DELIVERY_ZONES = {"beirut": 0, "hamra": 0, "achrafieh": 0, "jounieh": 10, "baabda": 10, "aley": 10, "dbayeh": 10,
                  "jbeil": 15, "byblos": 15, "tripoli": 15, "saida": 15, "sidon": 15, "zahle": 15,
                  "tyre": 20, "sour": 20, "nabatieh": 20, "baalbek": 20, "batroun": 15, "chouf": 15}
def estimate_delivery(city: str, rush: bool = False) -> str:
    key = city.lower().strip()
    build = "2 business days after parts arrive (rush, +$30)" if str(rush).lower() in ("true", "1", "yes") else \
            "3 to 5 business days after parts arrive"
    if key in DELIVERY_ZONES:
        fee = DELIVERY_ZONES[key]
        return f"Build time: {build}. Delivery to {city.title()}: {'free' if fee == 0 else f'${fee}'}, 1 to 2 days after the build."
    return (f"'{city}' is not in our delivery zones. Known prices exist only for Lebanon: free in Beirut, $10 to $20 elsewhere. "
            f"Build time: {build}.")

# ---------------- Tool 5 ----------------
DOC_TEXT = open("me/business_summary.txt", encoding="utf-8").read() + "\n\n" + \
           "\n".join(page.extract_text() or "" for page in PdfReader("me/about_business.pdf").pages)
CHUNKS = [c.strip() for c in re.split(r"\n\s*\n|(?<=\.)\n", DOC_TEXT) if len(c.strip()) > 40]
def lookup_policy(query: str) -> str:
    words = {w for w in re.findall(r"[a-z0-9]+", query.lower()) if len(w) > 2}
    scored = sorted(CHUNKS, key=lambda c: -sum(w in c.lower() for w in words))
    best = [c for c in scored[:2] if sum(w in c.lower() for w in words) > 0]
    return "\n---\n".join(best) if best else "No matching information in the RigSmith documents."

# ---------------- Tool 6 ----------------
os.makedirs("logs", exist_ok=True)
def record_lead(name: str, email: str, notes: str = "") -> str:
    lead = {"name": name, "email": email, "notes": notes, "time": datetime.datetime.now().isoformat(timespec="seconds")}
    with open("logs/leads.jsonl", "a", encoding="utf-8") as f:
        f.write(json.dumps(lead, ensure_ascii=False) + "\n")
    print(f"📥 [lead saved] {name} <{email}>")
    return f"Lead saved for {name} ({email}). The team will email a quote within 24 hours."

TOOLS = {"search_parts": search_parts, "price_build": price_build, "check_compatibility": check_compatibility,
         "estimate_delivery": estimate_delivery, "lookup_policy": lookup_policy, "record_lead": record_lead}

# Quick tool tests
print(search_parts("gpu", 500), "\n")
print(search_parts(["cpu", "psu"], 150), "\n")
print(check_compatibility(["cpu_r7_9700x", "mb_b550", "ram_32_ddr4", "psu_650", "gpu_rtx5070"]), "\n")
print(estimate_delivery("Tripoli", rush=True), "\n")
print(lookup_policy("installment payment"))

gpu_rtx5060: NVIDIA RTX 5060 8GB (1080p high) - $299 (watts=145, level=1080p)
gpu_rtx5060ti: NVIDIA RTX 5060 Ti 16GB (1440p / editing) - $449 (watts=180, level=1440p) 

[CPU]
cpu_r5_5600: AMD Ryzen 5 5600 (6 cores) - $109 (socket=AM4, watts=65)
cpu_i5_12400f: Intel Core i5-12400F (6 cores) - $119 (socket=LGA1700, watts=65)

[PSU]
psu_650: 650W 80+ Bronze PSU - $69 (watts=650)
psu_750: 750W 80+ Gold PSU - $99 (watts=750)
psu_850: 850W 80+ Gold PSU - $129 (watts=850) 

NOT COMPATIBLE
  OK: RAM DDR4 vs motherboard DDR4
  OK: PSU 650W vs recommended 500W
  PROBLEM: CPU socket AM5 vs motherboard socket AM4 

Build time: 2 business days after parts arrive (rush, +$30). Delivery to Tripoli: $15, 1 to 2 days after the build. 

Practical details: builds take 3 to 5 business days after parts arrive; a 48-hour rush option costs $30 extra. A 30% deposit is required to order parts. We accept cash (USD), Whish Money and card payments. Delivery is free inside Beirut and costs $10 to $20 elsewhere in 

## 3. Instructions block

Following the guidelines from class, the instructions are **separate from the persona**:

- each instruction has **one purpose**;
- the **inputs** (a customer message, then observations) and the **outputs** (the exact `Thought / Action / Action Input` or `Final Answer` format) are explicit;
- it contains a **worked example**;
- it says nothing about tone, so it **cannot conflict** with any persona. Tone and style live only in the persona.

The tool descriptions are embedded in the same block, so the model knows exactly which actions exist and what arguments they take.

In [4]:
TOOL_DESCRIPTIONS = '''AVAILABLE TOOLS (use exactly these names; Action Input must be valid JSON):
- search_parts: list catalog parts, cheapest first. "category" can be one category, a list of categories, or "all"
  (categories: cpu, motherboard, gpu, ram, storage, psu, case, cooler). Input: {"category": "gpu" | ["cpu", "gpu"] | "all", "max_price": number (optional)}
- price_build: itemized price of a list of part ids, plus assembly, the total, and the matching tier. Input: {"part_ids": ["id1", "id2", ...]}
- check_compatibility: checks CPU socket, RAM type, case size and PSU wattage. Input: {"part_ids": ["id1", "id2", ...]}
- estimate_delivery: build time and delivery fee for a city in Lebanon. Input: {"city": "name", "rush": true|false}
- lookup_policy: searches the RigSmith company documents (payments, warranty, hours, services). Input: {"query": "keywords"}
- record_lead: saves a customer who wants a quote. Only when you have their name AND email. Input: {"name": "...", "email": "...", "notes": "..."}'''

INSTRUCTIONS = '''INSTRUCTIONS
Input: you receive a customer message. After each Action you receive an "Observation:" with the tool result.
Output: every reply you write is EITHER one tool step OR the final answer, in exactly this format.

Tool step:
Thought: <one short sentence: what you need next and why>
Action: <tool name>
Action Input: <JSON object>

Final answer:
Thought: <one short sentence: why you can answer now>
Final Answer: <your answer to the customer>

Rules (one purpose each):
1. Write only ONE Action per reply, then stop and wait for its Observation. Never write an Observation yourself.
2. Every price you mention must come from a price_build or search_parts Observation.
3. Before recommending a full build, run check_compatibility on the exact part ids you recommend.
4. Keep a full build's TOTAL (from price_build) at or under the customer's budget.
5. Answer policy questions (payments, warranty, hours, services) only from a lookup_policy Observation.
6. If the request is too vague to act on (no budget or no use case), ask ONE clarifying question as your Final Answer.
7. Call record_lead only after the customer has given both a name and an email.
8. For a full build, get all the categories you need with ONE search_parts call (a list or "all"), not one call per category.
9. Use at most 8 tool steps, then give your best Final Answer.

WORKED EXAMPLE
Customer: Will the Ryzen 5 5600 work with the B650 board?
Thought: I need to check the socket of both parts.
Action: check_compatibility
Action Input: {"part_ids": ["cpu_r5_5600", "mb_b650"]}
Observation: NOT COMPATIBLE
  PROBLEM: CPU socket AM4 vs motherboard socket AM5
Thought: The sockets differ, so I can answer.
Final Answer: No. The Ryzen 5 5600 uses the AM4 socket and the B650 board is AM5, so they don't fit together. Pair the B650 with an AM5 CPU like the Ryzen 5 7600, or keep the 5600 with our B550 board.'''

COT_FORMAT_NOTE = "Always start each reply with a 'Thought:' line before the Action or Final Answer."
NO_COT_FORMAT_NOTE = ("Do NOT write any 'Thought:' lines. Reply with only 'Action:' + 'Action Input:', "
                      "or only 'Final Answer:'.")
print(INSTRUCTIONS[:600], "...")

INSTRUCTIONS
Input: you receive a customer message. After each Action you receive an "Observation:" with the tool result.
Output: every reply you write is EITHER one tool step OR the final answer, in exactly this format.

Tool step:
Thought: <one short sentence: what you need next and why>
Action: <tool name>
Action Input: <JSON object>

Final answer:
Thought: <one short sentence: why you can answer now>
Final Answer: <your answer to the customer>

Rules (one purpose each):
1. Write only ONE Action per reply, then stop and wait for its Observation. Never write an Observation yourself.
2. Every ...


## 4. Personas: prompt traits and LLM configuration

Each persona has a different **task focus** and is defined in two ways, as covered in class:

1. **Prompt traits:** tone, vocabulary, response style, and values.
2. **A matching LLM configuration.** Each setting is chosen *for* the persona; the reasons are in the table below.

| Persona | Task focus | temperature | top_p | top_k | presence / frequency penalty | max tokens | Why this configuration |
|---|---|---|---|---|---|---|---|
| **Rex, Friendly Advisor** | first-time buyers: recommends complete builds | 0.8 | 0.95 | 40 | 0.3 / 0.0 | 900 | Higher temperature and top-p allow warm, varied wording. A small **presence** penalty nudges him to mention new options instead of repeating one. |
| **Volt, Strict Expert** | enthusiasts: technical compatibility and spec questions | 0.1 | 0.5 | 10 | 0.0 / 0.0 | 600 | Near-deterministic sampling (low temperature, narrow top-p/top-k) for exact, repeatable technical answers. No penalties, because repeating exact part names and numbers is *desired*. Shorter max tokens for terse answers. |
| **Maya, Cautious Helper** | budget-protective buyers: verifies everything, flags risks, asks before assuming | 0.3 | 0.8 | 20 | 0.0 / 0.3 | 800 | Low-moderate temperature for careful, consistent answers. A small **frequency** penalty stops her caveats from repeating the same warning phrases. |

The **instructions block is identical for all three personas**, so any difference in behavior comes from the persona prompt and the configuration.

In [5]:
PERSONAS = {
    "Rex (Friendly Advisor)": {
        "prompt": '''PERSONA: You are Rex, the friendly build advisor at RigSmith PCs in Hamra, Beirut.
Tone: warm, upbeat, encouraging, like a gamer friend who knows hardware.
Vocabulary: simple words; explain any technical term in a few words; an occasional gaming expression is fine.
Response style: 3 to 6 sentences; lead with the recommendation, then the total, then one friendly next step
(for example, offering to send a quote).
Values: make PC building feel easy and exciting; never make the customer feel dumb.''',
        "config": {"temperature": 0.8, "top_p": 0.95, "top_k": 40, "presence_penalty": 0.3, "frequency_penalty": 0.0,
                   "max_output_tokens": 900},
    },
    "Volt (Strict Expert)": {
        "prompt": '''PERSONA: You are Volt, RigSmith PCs' senior hardware engineer.
Tone: precise, direct, no small talk, no emojis.
Vocabulary: exact technical terms (socket, chipset, DDR generation, wattage headroom) with exact part names and numbers.
Response style: short answer first, then the facts as compact lines (part: price, spec). No filler sentences.
Values: technical correctness above everything; state limitations plainly.''',
        "config": {"temperature": 0.1, "top_p": 0.5, "top_k": 10, "presence_penalty": 0.0, "frequency_penalty": 0.0,
                   "max_output_tokens": 600},
    },
    "Maya (Cautious Helper)": {
        "prompt": '''PERSONA: You are Maya, RigSmith PCs' careful build consultant who protects customers' budgets.
Tone: calm, patient, reassuring.
Vocabulary: plain language; point out risks clearly ("one thing to watch out for...").
Response style: confirm what you understood, give the verified answer, then flag any risk or assumption you made.
Values: never let a customer buy something incompatible or over budget; when unsure, ask instead of assuming;
prefer leaving some budget headroom.''',
        "config": {"temperature": 0.3, "top_p": 0.8, "top_k": 20, "presence_penalty": 0.0, "frequency_penalty": 0.3,
                   "max_output_tokens": 800},
    },
}
for name, p in PERSONAS.items():
    print(f"{name}: {p['config']}")

Rex (Friendly Advisor): {'temperature': 0.8, 'top_p': 0.95, 'top_k': 40, 'presence_penalty': 0.3, 'frequency_penalty': 0.0, 'max_output_tokens': 900}
Volt (Strict Expert): {'temperature': 0.1, 'top_p': 0.5, 'top_k': 10, 'presence_penalty': 0.0, 'frequency_penalty': 0.0, 'max_output_tokens': 600}
Maya (Cautious Helper): {'temperature': 0.3, 'top_p': 0.8, 'top_k': 20, 'presence_penalty': 0.0, 'frequency_penalty': 0.3, 'max_output_tokens': 800}


## 5. The ReAct loop in LangGraph (implemented by hand)

```text
          ┌─────────────────────────────────────────────┐
          ▼                                             │
 START → think ──(Action detected)──► act ──(Observation appended)
          │
          ├──(format error) ──► think   (an error observation is sent back to the model)
          │
          └──(Final Answer, or step limit reached) ──► finish → END
```

- **`think` node:** calls Gemini with the persona, the instructions, the tool descriptions and the scratchpad so far. The call uses the stop sequence `Observation:`, so the model cannot invent its own observations.
- **Action detection:** `parse_reply()` uses regular expressions to find `Final Answer:` or `Action:` + `Action Input:` (JSON). If both appear, whichever comes first in the text wins. If the output matches neither, it counts as a format error, and an error observation tells the model the correct format.
- **`act` node:** runs the Python tool with the parsed JSON arguments, catches tool errors, and appends `Observation: ...` to the scratchpad.
- **Conditional edges:** after `think`, the graph routes to `act` (an action was found), back to `think` (a format error), or to `finish` (a final answer, or the step limit of 10 model calls).
- **`finish` node:** returns the final answer. If the step limit was hit, it returns a safe fallback message.

The scratchpad is a list of Gemini messages: the model's `Thought/Action` turns alternate with `Observation` turns. Earlier conversation turns (question + final answer only) are included as short-term memory.

In [6]:
from langgraph.graph import StateGraph, START, END

MAX_STEPS = 10      # LLM calls per question (8 tool steps + final answer + margin for format errors)
PREFERRED_MODELS = ["gemini-3.6-flash", "gemini-3.5-flash", "gemini-3-flash-preview", "gemini-2.5-flash",
                    "gemini-3.5-flash-lite", "gemini-3.1-flash-lite", "gemini-2.5-flash-lite"]
try:
    _names = {m.name.split("/")[-1] for m in client.models.list()}
    MODELS = [m for m in PREFERRED_MODELS if m in _names] or PREFERRED_MODELS
except Exception:
    MODELS = PREFERRED_MODELS

def probe_models(models):
    # One tiny call per model: keep only the models that answer right now (free-tier quotas differ per model).
    working = []
    for m in models:
        try:
            client.models.generate_content(model=m, contents="Reply with OK.",
                                           config=types.GenerateContentConfig(max_output_tokens=200))
            working.append(m); print(f"  ✅ {m}")
        except Exception as e:
            print(f"  ❌ {m}: {str(e)[:70]}")
        time.sleep(2)
    return working

print("Testing which Gemini models respond right now:")
WORKING_MODELS = probe_models(MODELS) or MODELS
DEFAULT_MODEL = next((m for m in WORKING_MODELS if "lite" not in m), WORKING_MODELS[0])
LITE_MODEL = next((m for m in WORKING_MODELS if "lite" in m and m != DEFAULT_MODEL), None)
print("Default model:", DEFAULT_MODEL, "| lite model:", LITE_MODEL)

PACE_SECONDS = 4      # pause between API calls to stay under free-tier requests-per-minute limits
COOLDOWN = {}         # model -> time until which it is skipped after a rate-limit error

def call_llm(model, contents, system, config):
    # One Gemini call. If the requested model is busy or rate-limited, switch to another working model; wait only when
    # ALL models are busy. If a model rejects a parameter (e.g. penalties), drop it and log it.
    # Returns (text, tokens, dropped_params, model_actually_used).
    cfg, dropped = dict(config), []
    order = [model] + [m for m in WORKING_MODELS if m != model]
    order = [m for m in order if COOLDOWN.get(m, 0) < time.time()] or order   # skip models that were just rate-limited
    for attempt in range(4):
        i = 0
        while i < len(order):
            m = order[i]
            try:
                time.sleep(PACE_SECONDS)
                response = client.models.generate_content(
                    model=m, contents=contents,
                    config=types.GenerateContentConfig(system_instruction=system, stop_sequences=["Observation:"], **cfg))
                usage = getattr(response, "usage_metadata", None)
                return (response.text or "").strip(), getattr(usage, "total_token_count", 0) or 0, dropped, m
            except Exception as e:
                err = str(e)
                bad = [k for k in ("presence_penalty", "frequency_penalty", "top_k") if k in cfg and
                       (k in err or ("enalty" in err and "penalty" in k))]
                if "400" in err and bad:                       # unsupported parameter: drop it and retry the same model
                    for k in bad:
                        cfg.pop(k); dropped.append(k)
                    continue
                if any(c in err for c in ("429", "500", "503", "RESOURCE_EXHAUSTED", "UNAVAILABLE", "INTERNAL")):
                    COOLDOWN[m] = time.time() + 90                 # rest this model for 90 s
                    print(f"   [api] {m} busy/rate-limited ({err[:3]}), trying another model")
                    i += 1; continue
                raise
        wait = 20 * (attempt + 1)
        print(f"   [api] all models busy, waiting {wait}s"); time.sleep(wait)
    raise RuntimeError("Gemini kept failing after several retries")

# ---------------- Action detection ----------------
FINAL_RE  = re.compile(r"Final Answer\s*:\s*(.*)", re.S | re.I)
ACTION_RE = re.compile(r"Action\s*:\s*`?([a-zA-Z_]+)`?\s*\n?\s*Action Input\s*:\s*(\{.*\})", re.S | re.I)

def parse_reply(text):
    # Returns ("final", answer) | ("action", tool, args) | ("error", message)
    f, a = FINAL_RE.search(text), ACTION_RE.search(text)
    if a and (not f or a.start() < f.start()):
        tool, raw = a.group(1), a.group(2)
        raw = raw[: raw.rfind("}") + 1]
        try:
            args = json.loads(raw)
        except json.JSONDecodeError as e:
            return ("error", f"Action Input is not valid JSON ({e.msg}). Write it as a JSON object.")
        if tool not in TOOLS:
            return ("error", f"Unknown tool '{tool}'. Use one of: {', '.join(TOOLS)}.")
        return ("action", tool, args)
    if f:
        return ("final", f.group(1).strip())
    return ("error", "Your reply had no 'Action:' + 'Action Input:' and no 'Final Answer:'. Follow the format exactly.")

def thought_of(text):
    m = re.search(r"Thought\s*:\s*(.*?)(?:\n|$)", text, re.I)
    return m.group(1).strip() if m else ""

print(parse_reply('Thought: check\nAction: check_compatibility\nAction Input: {"part_ids": ["a", "b"]}'))
print(parse_reply("Thought: done\nFinal Answer: Yes, they fit."))
print(parse_reply("I think they fit."))

Testing which Gemini models respond right now:
  ✅ gemini-3.6-flash
  ✅ gemini-3.5-flash
  ✅ gemini-3-flash-preview
  ❌ gemini-2.5-flash: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/g
  ✅ gemini-3.5-flash-lite
  ✅ gemini-3.1-flash-lite
  ❌ gemini-2.5-flash-lite: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/g
Default model: gemini-3.6-flash | lite model: gemini-3.5-flash-lite
('action', 'check_compatibility', {'part_ids': ['a', 'b']})
('final', 'Yes, they fit.')
('error', "Your reply had no 'Action:' + 'Action Input:' and no 'Final Answer:'. Follow the format exactly.")


In [7]:
class AgentState(TypedDict):
    question: str
    history: list            # earlier turns: [{"role": "user"|"assistant", "content": str}]
    persona: str
    variant: dict            # {"cot": bool, "few_shot": bool, "in_message": bool}
    config: dict             # LLM sampling configuration
    model: str
    scratchpad: list         # Gemini Content objects for this question's ReAct loop
    pending: Optional[tuple] # (tool, args) waiting to be executed
    final: Optional[str]
    steps: int
    parse_errors: int
    tokens: int
    trace: list              # readable log of every step
    dropped_params: list
    models_used: list        # the model that actually answered each step

def build_prompt(persona, variant):
    parts = [PERSONAS[persona]["prompt"], INSTRUCTIONS if variant["few_shot"] else INSTRUCTIONS.split("WORKED EXAMPLE")[0],
             TOOL_DESCRIPTIONS, COT_FORMAT_NOTE if variant["cot"] else NO_COT_FORMAT_NOTE]
    return "\n\n".join(parts)

def start_node(state):
    prompt = build_prompt(state["persona"], state["variant"])
    contents = [types.Content(role="model" if m["role"] == "assistant" else "user", parts=[types.Part(text=m["content"])])
                for m in state["history"][-6:]]
    question = state["question"]
    if state["variant"]["in_message"]:                       # experiment: instructions inside the user message
        question = f"{prompt}\n\n---\nCustomer message: {question}"
    contents.append(types.Content(role="user", parts=[types.Part(text=question)]))
    return {"scratchpad": contents, "steps": 0, "parse_errors": 0, "tokens": 0, "trace": [], "final": None,
            "pending": None, "dropped_params": [], "models_used": []}

def think_node(state):
    system = None if state["variant"]["in_message"] else build_prompt(state["persona"], state["variant"])
    text, tokens, dropped, used = call_llm(state["model"], state["scratchpad"], system, state["config"])
    pad = state["scratchpad"] + [types.Content(role="model", parts=[types.Part(text=text or "(empty)")])]
    parsed = parse_reply(text)
    trace = state["trace"] + [{"type": "llm", "text": text, "thought": thought_of(text)}]
    update = {"scratchpad": pad, "tokens": state["tokens"] + tokens, "steps": state["steps"] + 1,
              "dropped_params": sorted(set(state["dropped_params"]) | set(dropped)), "pending": None,
              "models_used": state["models_used"] + [used]}
    if parsed[0] == "final":
        update["final"] = parsed[1]
    elif parsed[0] == "action":
        update["pending"] = (parsed[1], parsed[2])
    else:                                                    # format error: tell the model and loop back
        pad.append(types.Content(role="user", parts=[types.Part(text=f"Observation: FORMAT ERROR. {parsed[1]}")]))
        update["parse_errors"] = state["parse_errors"] + 1
        trace.append({"type": "error", "text": parsed[1]})
    update["trace"] = trace
    return update

def act_node(state):
    tool, args = state["pending"]
    try:
        observation = TOOLS[tool](**args)
    except TypeError as e:
        observation = f"Tool error: wrong arguments for {tool} ({e}). Check the Action Input format."
    except Exception as e:
        observation = f"Tool error: {e}"
    pad = state["scratchpad"] + [types.Content(role="user", parts=[types.Part(text=f"Observation: {observation}")])]
    trace = state["trace"] + [{"type": "tool", "tool": tool, "args": args, "observation": observation}]
    return {"scratchpad": pad, "trace": trace, "pending": None}

def finish_node(state):
    if state["final"]:
        return {}
    return {"final": "Sorry, I couldn't finish that within my step limit. Could you rephrase or narrow it down?"}

def route_after_think(state):
    if state["final"]:
        return "finish"
    if state["steps"] >= MAX_STEPS:
        return "finish"
    return "act" if state["pending"] else "think"

graph = StateGraph(AgentState)
graph.add_node("start", start_node)
graph.add_node("think", think_node)
graph.add_node("act", act_node)
graph.add_node("finish", finish_node)
graph.add_edge(START, "start")
graph.add_edge("start", "think")
graph.add_conditional_edges("think", route_after_think, {"act": "act", "think": "think", "finish": "finish"})
graph.add_edge("act", "think")
graph.add_edge("finish", END)
react_app = graph.compile()

print(react_app.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	start(start)
	think(think)
	act(act)
	finish(finish)
	__end__([<p>__end__</p>]):::last
	__start__ --> start;
	act --> think;
	start --> think;
	think -.-> act;
	think -.-> finish;
	finish --> __end__;
	think -.-> think;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [8]:
DEFAULT_VARIANT = {"cot": True, "few_shot": True, "in_message": False}

def run_agent(question, persona="Rex (Friendly Advisor)", variant=None, config=None, model=None, history=None, verbose=True):
    state = {"question": question, "history": history or [], "persona": persona,
             "variant": {**DEFAULT_VARIANT, **(variant or {})},
             "config": config or PERSONAS[persona]["config"], "model": model or DEFAULT_MODEL}
    t0 = time.time()
    result = react_app.invoke(state, config={"recursion_limit": 40})
    result["latency"] = round(time.time() - t0, 1)
    if verbose:
        print(f"\n🧑 {question}\n🎭 {persona} | requested model={state['model']} | used={sorted(set(result['models_used']))}")
        for step in result["trace"]:
            if step["type"] == "llm":
                print("   " + step["text"].replace("\n", "\n   "))
            elif step["type"] == "tool":
                print("   👀 Observation: " + step["observation"].replace("\n", "\n      "))
            else:
                print("   ⚠️ " + step["text"])
        print(f"✅ FINAL: {result['final']}\n   (steps={result['steps']}, tokens={result['tokens']}, {result['latency']}s)")
    return result

_ = run_agent("Will the Ryzen 7 9700X work with the B550 motherboard?", persona="Volt (Strict Expert)")


🧑 Will the Ryzen 7 9700X work with the B550 motherboard?
🎭 Volt (Strict Expert) | requested model=gemini-3.6-flash | used=['gemini-3.6-flash']
   Thought: I need to find the exact part IDs for the Ryzen 7 9700X and a B550 motherboard to check compatibility.
   Action: search_parts
   Action Input: {"category": ["cpu", "motherboard"]}
   👀 Observation: [CPU]
      cpu_r5_5600: AMD Ryzen 5 5600 (6 cores) - $109 (socket=AM4, watts=65)
      cpu_i5_12400f: Intel Core i5-12400F (6 cores) - $119 (socket=LGA1700, watts=65)
      cpu_r5_7600: AMD Ryzen 5 7600 (6 cores) - $189 (socket=AM5, watts=65)
      cpu_r5_9600x: AMD Ryzen 5 9600X (6 cores) - $229 (socket=AM5, watts=65)
      cpu_i5_14600k: Intel Core i5-14600K (14 cores) - $279 (socket=LGA1700, watts=125)
      cpu_r7_9700x: AMD Ryzen 7 9700X (8 cores) - $329 (socket=AM5, watts=65)
      cpu_r7_7800x3d: AMD Ryzen 7 7800X3D (8 cores, gaming) - $399 (socket=AM5, watts=120)
      
      [MOTHERBOARD]
      mb_b550: B550 ATX motherboard (AM

## 6. Experiments

### Test tasks

Five tasks, each with an **automatic success check**, so every run can be scored the same way:

| ID | Task | What counts as success |
|---|---|---|
| T1 | Full build: "$1,300 for 1440p gaming and streaming, check compatibility, give the total" | `price_build` **and** `check_compatibility` were called, the last priced TOTAL is ≤ $1,300, and that total appears in the answer |
| T2 | Compatibility: "Ryzen 7 9700X + B550?" | the answer says it is **not** compatible |
| T3 | Delivery: "rush build delivered to Tripoli: cost and time?" | the answer mentions **$15** and the rush time (2 days / 48h) |
| T4 | Policy: "Can I pay in installments?" | `lookup_policy` was called and the answer says **no** installments |
| T5 | Vague: "I want a good PC." | the answer asks a clarifying **question** (and has no invented price) |

### Experiment design

1. **E1, personas:** 3 personas × 5 tasks, each with its own configuration (CoT on, few-shot, system prompt).
2. **E2, prompt engineering** (persona Rex, tasks T1 + T2):
   - without Chain-of-Thought;
   - zero-shot (no worked example);
   - instructions in the user message instead of the system prompt.
3. **E3, configuration** (persona Volt's prompt, tasks T1 + T2):
   - temperature 0.0 vs 1.0 vs 1.6;
   - narrow top-p/top-k;
   - high penalties;
   - a small max-token limit;
   - a lite model instead of the default.

Every run is appended to `logs/experiments.jsonl` (question, settings, full trace, metrics), so nothing is lost and re-running continues where it stopped.

In [9]:
TASKS = {
    "T1": "I have $1,300 for a PC for 1440p gaming and streaming. Pick the parts, check they're compatible, and give me the total.",
    "T2": "Will the Ryzen 7 9700X work with the B550 motherboard?",
    "T3": "If I order a rush build, how much will delivery to Tripoli cost and how long will it take?",
    "T4": "Can I pay for my PC in installments?",
    "T5": "I want a good PC.",
}

def money(text):
    return [int(x.replace(",", "")) for x in re.findall(r"\$\s?(\d[\d,]*)", text)]

def check_success(task, result):
    answer, trace = result["final"] or "", result["trace"]
    tools = [s["tool"] for s in trace if s["type"] == "tool"]
    low = answer.lower()
    if task == "T1":
        totals = [int(m) for s in trace if s["type"] == "tool" and s["tool"] == "price_build"
                  for m in re.findall(r"TOTAL: \$(\d+)", s["observation"])]
        return bool(totals) and "check_compatibility" in tools and totals[-1] <= 1300 and totals[-1] in money(answer)
    if task == "T2":
        return bool(re.search(r"\b(no|not compatible|incompatible|won't|will not|doesn't|does not|can't|cannot)\b", low))
    if task == "T3":
        return "15" in answer and bool(re.search(r"2 (business )?days|48|two days", low))
    if task == "T4":
        return "lookup_policy" in tools and bool(re.search(r"\b(no|not|don't|do not)\b", low)) and "installment" in low
    if task == "T5":
        return "?" in answer and not money(answer)
    return False

LOG = "logs/experiments.jsonl"
# To start the experiments over (e.g. after changing the agent), delete the log:  os.remove(LOG)
def load_log():
    if not os.path.exists(LOG):
        return []
    return [json.loads(l) for l in open(LOG, encoding="utf-8") if l.strip()]

def run_experiment(exp, label, task, persona, variant=None, config=None, model=None):
    key = f"{exp}|{label}|{task}"
    if any(r["key"] == key for r in load_log()):
        print(f"skip (already done): {key}"); return
    try:
        res = run_agent(TASKS[task], persona=persona, variant=variant, config=config, model=model, verbose=False)
    except Exception as e:
        print(f"FAILED {key}: {str(e)[:100]}"); return
    tools = [s["tool"] for s in res["trace"] if s["type"] == "tool"]
    row = {"key": key, "experiment": exp, "setup": label, "task": task, "persona": persona,
           "variant": {**DEFAULT_VARIANT, **(variant or {})}, "config": config or PERSONAS[persona]["config"],
           "model": model or DEFAULT_MODEL, "success": check_success(task, res), "steps": res["steps"],
           "tool_calls": len(tools), "tools": tools, "parse_errors": res["parse_errors"], "tokens": res["tokens"],
           "latency_s": res["latency"], "answer_words": len((res["final"] or "").split()),
           "dropped_params": res["dropped_params"], "models_used": sorted(set(res["models_used"])),
           "answer": res["final"], "trace": res["trace"]}
    with open(LOG, "a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"{'✅' if row['success'] else '❌'} {key:45s} steps={row['steps']} tools={tools} tokens={row['tokens']}")

In [10]:
# E1: personas x tasks (each persona with its own configuration)
for persona in PERSONAS:
    for task in TASKS:
        run_experiment("E1", persona, task, persona)

   [api] gemini-3.6-flash busy/rate-limited (429), trying another model
   [api] gemini-3.5-flash busy/rate-limited (429), trying another model
   [api] gemini-3-flash-preview busy/rate-limited (429), trying another model


/usr/local/lib/python3.13/dist-packages/google/genai/_common.py:651: UserWarning: MALFORMED_RESPONSE is not a valid FinishReason
  warnings.warn(f'{value} is not a valid {cls.__name__}')


✅ E1|Rex (Friendly Advisor)|T1                  steps=9 tools=['search_parts', 'price_build', 'price_build', 'price_build', 'check_compatibility'] tokens=19215
   [api] gemini-3.6-flash busy/rate-limited (429), trying another model
   [api] gemini-3.5-flash busy/rate-limited (429), trying another model
   [api] gemini-3-flash-preview busy/rate-limited (429), trying another model
✅ E1|Rex (Friendly Advisor)|T2                  steps=3 tools=['check_compatibility'] tokens=3330
✅ E1|Rex (Friendly Advisor)|T3                  steps=3 tools=['estimate_delivery'] tokens=3130
   [api] gemini-3.6-flash busy/rate-limited (429), trying another model
   [api] gemini-3.5-flash busy/rate-limited (429), trying another model
   [api] gemini-3-flash-preview busy/rate-limited (429), trying another model
✅ E1|Rex (Friendly Advisor)|T4                  steps=8 tools=['lookup_policy', 'lookup_policy'] tokens=9139
✅ E1|Rex (Friendly Advisor)|T5                  steps=1 tools=[] tokens=1131
✅ E1|Volt (Stric

In [11]:
# E2: prompt engineering (persona Rex, tasks T1 and T2). The baseline is E1's Rex run.
E2_VARIANTS = {
    "baseline (CoT, few-shot, system)": {},
    "no Chain-of-Thought":              {"cot": False},
    "zero-shot (no worked example)":    {"few_shot": False},
    "instructions in user message":     {"in_message": True},
}
for label, variant in E2_VARIANTS.items():
    if not variant:
        continue                       # baseline already logged in E1
    for task in ["T1", "T2"]:
        run_experiment("E2", label, task, "Rex (Friendly Advisor)", variant=variant)

❌ E2|no Chain-of-Thought|T1                     steps=5 tools=['search_parts', 'price_build', 'price_build', 'check_compatibility'] tokens=11614
   [api] gemini-3.6-flash busy/rate-limited (429), trying another model
   [api] gemini-3.5-flash busy/rate-limited (429), trying another model
✅ E2|no Chain-of-Thought|T2                     steps=2 tools=['check_compatibility'] tokens=2146
   [api] gemini-3-flash-preview busy/rate-limited (429), trying another model
   [api] gemini-3.6-flash busy/rate-limited (429), trying another model
   [api] gemini-3.5-flash busy/rate-limited (429), trying another model
   [api] gemini-3-flash-preview busy/rate-limited (429), trying another model
✅ E2|zero-shot (no worked example)|T1           steps=5 tools=['search_parts', 'price_build', 'price_build', 'check_compatibility'] tokens=11389
❌ E2|zero-shot (no worked example)|T2           steps=4 tools=['check_compatibility'] tokens=3620
   [api] gemini-3.6-flash busy/rate-limited (429), trying another mode

In [12]:
# E3: configuration sweep (Volt's prompt, tasks T1 and T2)
base = PERSONAS["Volt (Strict Expert)"]["config"]
E3_CONFIGS = {
    "temp 0.0":                   ({**base, "temperature": 0.0}, None),
    "temp 1.0":                   ({**base, "temperature": 1.0, "top_p": 0.95, "top_k": 40}, None),
    "temp 1.6 (very random)":     ({**base, "temperature": 1.6, "top_p": 1.0, "top_k": 64}, None),
    "top_p 0.2 / top_k 3":        ({**base, "top_p": 0.2, "top_k": 3}, None),
    "high penalties (1.0 / 1.0)": ({**base, "presence_penalty": 1.0, "frequency_penalty": 1.0}, None),
    "max_tokens 120":             ({**base, "max_output_tokens": 120}, None),
}
if LITE_MODEL:
    E3_CONFIGS[f"lite model ({LITE_MODEL})"] = (base, LITE_MODEL)
for label, (cfg, model) in E3_CONFIGS.items():
    for task in ["T1", "T2"]:
        run_experiment("E3", label, task, "Volt (Strict Expert)", config=cfg, model=model)

   [api] gemini-3.6-flash busy/rate-limited (429), trying another model
   [api] gemini-3.5-flash busy/rate-limited (429), trying another model
   [api] gemini-3-flash-preview busy/rate-limited (429), trying another model
✅ E3|temp 0.0|T1                                steps=7 tools=['search_parts', 'check_compatibility', 'price_build', 'price_build', 'check_compatibility'] tokens=16198
✅ E3|temp 0.0|T2                                steps=4 tools=['check_compatibility'] tokens=4179
   [api] gemini-3.6-flash busy/rate-limited (429), trying another model
   [api] gemini-3.5-flash busy/rate-limited (429), trying another model
✅ E3|temp 1.0|T1                                steps=8 tools=['search_parts', 'check_compatibility', 'price_build', 'price_build', 'check_compatibility'] tokens=18436
   [api] gemini-3-flash-preview busy/rate-limited (429), trying another model
✅ E3|temp 1.0|T2                                steps=4 tools=['check_compatibility'] tokens=4202
✅ E3|temp 1.6 (very rand

## 7. Results

The tables below are computed from `logs/experiments.jsonl`.

In [13]:
df = pd.DataFrame(load_log())
cols = ["success", "steps", "tool_calls", "parse_errors", "tokens", "latency_s", "answer_words"]
pd.set_option("display.max_colwidth", 120)

print("E1: personas (each with its own configuration)")
e1 = df[df.experiment == "E1"]
display(e1.pivot_table(index="persona", columns="task", values="success", aggfunc="first"))
display(e1.groupby("persona")[cols].mean().round(2))

print("\nE2: prompt engineering (Rex; T1 + T2). Baseline = E1 Rex rows")
base_rows = e1[(e1.persona == "Rex (Friendly Advisor)") & e1.task.isin(["T1", "T2"])].assign(setup="baseline (CoT, few-shot, system)")
e2 = pd.concat([base_rows, df[df.experiment == "E2"]])
display(e2.groupby("setup")[cols].mean().round(2))

print("\nE3: configurations (Volt's prompt; T1 + T2). Baseline = E1 Volt rows")
base_rows = e1[(e1.persona == "Volt (Strict Expert)") & e1.task.isin(["T1", "T2"])].assign(setup="Volt default (temp 0.1)")
e3 = pd.concat([base_rows, df[df.experiment == "E3"]])
display(e3.groupby("setup")[cols + ["dropped_params", "models_used"]].agg(
    {**{c: "mean" for c in cols}, "dropped_params": lambda x: sorted({p for l in x for p in l}),
     "models_used": lambda x: sorted({p for l in x for p in l})}).round(2))

E1: personas (each with its own configuration)


task,T1,T2,T3,T4,T5
persona,,,,,
Maya (Cautious Helper),True,True,True,True,True
Rex (Friendly Advisor),True,True,True,True,True
Volt (Strict Expert),True,True,True,True,False


,success,steps,tool_calls,parse_errors,tokens,latency_s,answer_words
persona,,,,,,,
Maya (Cautious Helper),1.0,4.6,1.4,2.2,6647.2,53.44,75.0
Rex (Friendly Advisor),1.0,4.8,1.8,2.0,7189.0,54.66,51.8
Volt (Strict Expert),0.8,3.8,1.2,1.6,5062.8,23.94,33.8



E2: prompt engineering (Rex; T1 + T2). Baseline = E1 Rex rows


,success,steps,tool_calls,parse_errors,tokens,latency_s,answer_words
setup,,,,,,,
"baseline (CoT, few-shot, system)",1.0,6.0,3.0,2.0,11272.5,70.40,54.5
instructions in user message,1.0,5.5,3.0,1.5,8915.0,69.85,59.5
no Chain-of-Thought,0.5,3.5,2.5,0.0,6880.0,35.75,57.5
zero-shot (no worked example),0.5,4.5,2.5,1.0,7504.5,96.80,72.0



E3: configurations (Volt's prompt; T1 + T2). Baseline = E1 Volt rows


,success,steps,tool_calls,parse_errors,tokens,latency_s,answer_words,dropped_params,models_used
setup,,,,,,,,,
Volt default (temp 0.1),1.0,5.5,2.0,2.5,8398.5,36.85,50.5,[],"[gemini-3-flash-preview, gemini-3.5-flash-lite]"
high penalties (1.0 / 1.0),1.0,5.0,2.5,1.5,8701.0,54.80,44.5,"[frequency_penalty, presence_penalty]",[gemini-3.5-flash-lite]
lite model (gemini-3.5-flash-lite),1.0,4.0,3.0,0.0,8416.0,134.95,43.0,[],"[gemini-3.1-flash-lite, gemini-3.5-flash-lite]"
max_tokens 120,0.5,4.0,2.5,0.5,7491.5,30.00,23.5,[],"[gemini-3.1-flash-lite, gemini-3.5-flash-lite]"
temp 0.0,1.0,5.5,3.0,1.5,10188.5,35.75,44.0,[],[gemini-3.5-flash-lite]
temp 1.0,1.0,6.0,3.0,2.0,11319.0,38.70,56.0,[],"[gemini-3-flash-preview, gemini-3.5-flash-lite]"
temp 1.6 (very random),1.0,5.5,3.0,1.5,10062.0,33.15,44.0,[],[gemini-3.5-flash-lite]
top_p 0.2 / top_k 3,1.0,5.5,3.0,1.5,10382.5,31.80,49.0,[],[gemini-3.5-flash-lite]


In [14]:
# Side-by-side answers: the same task, three personas
for task in ["T1", "T4", "T5"]:
    print("=" * 100, f"\n{task}: {TASKS[task]}")
    for _, r in e1[e1.task == task].iterrows():
        print(f"\n--- {r.persona} ({'✅' if r.success else '❌'}) ---\n{r.answer}")

T1: I have $1,300 for a PC for 1440p gaming and streaming. Pick the parts, check they're compatible, and give me the total.

--- Rex (Friendly Advisor) (✅) ---
This killer setup features the Ryzen 5 7600 and an RTX 5070, which is fantastic for smooth 1440p gaming and crisp streaming! The total comes out to $1,278 including professional assembly and our 48-hour stress test. Shall I send you a detailed quote to lock this in?

--- Volt (Strict Expert) (✅) ---
cpu_r5_7600: $189, AMD Ryzen 5 7600 (socket=AM5, watts=65)
mb_b650m: $129, B650M micro-ATX motherboard (socket=AM5, ram=DDR5, form=mATX)
gpu_rtx5060ti: $449, NVIDIA RTX 5060 Ti 16GB (watts=180, level=1440p)
ram_32_ddr5: $99, 32GB DDR5-6000 (type=DDR5)
ssd_1tb: $69, 1TB NVMe SSD
psu_650: $69, 650W 80+ Bronze PSU (watts=650)
case_matx: $65, Compact case (fits mATX)
cooler_air: $39, Dual-tower air cooler
Assembly: $40
TOTAL: $1148

--- Maya (Cautious Helper) (✅) ---
I understand you are looking for a reliable 1440p gaming and streaming 

### Reflection

*All numbers come from the tables above (`logs/experiments.jsonl`).*

**1. Which persona gave the most helpful or natural results?**
All three personas solved T1–T4 (full build, compatibility, delivery, policy). The differences were in style and efficiency:

- **Rex (Friendly Advisor):** 5/5. The most natural for first-time buyers. He was warm and short (about 52 words per answer) and always ended with a next step. On T5 he asked about budget and games in a friendly way.
- **Maya (Cautious Helper):** 5/5. The most *helpful* on the full build. She explained that she left budget headroom ("\$1,278, leaving you \$22 safely under your \$1,300 budget") and confirmed the socket, RAM and power checks. Her answers were also the longest (about 75 words).
- **Volt (Strict Expert):** 4/5, but by far the most efficient: 3.8 steps, about 5,100 tokens and 24 s per task, compared with about 54 s for the other two. His T1 answer was a clean parts list with a \$1,148 total. His only miss was T5: he replied *"State your exact budget in USD and your primary use case"*. That does ask for the missing information, but as a command, so my checker (which looks for a "?") marked it wrong. It also shows that a strict tone feels cold to a vague first-time customer.

**Conclusion:** Maya is best for full builds where budget matters, Volt for quick technical checks, and Rex for a friendly first contact. For RigSmith's main job, planning a build, **Maya** gave the most helpful result.

**2. Which prompt and configuration combination performed best?**
The best prompt setup was the **baseline: Chain-of-Thought + few-shot (worked example) + system prompt**, at 2/2 in E2. The two variants that removed something each dropped to 1/2. Putting the instructions in the user message also scored 2/2 and used fewer tokens (8,915 vs 11,273), but the system prompt is cleaner because the persona doesn't have to be repeated in every message.

For configuration, Volt's **low temperature (0.1) with narrow top-p/top-k** gave the fewest steps and tokens in E1. In E3, temperature 0.0, 1.0 and 1.6 all still passed, so temperature mattered less than I expected. The tool observations seem to anchor the answers, so the model can't drift far even at high randomness.

The setting that clearly broke something was **max_tokens = 120**: it failed the full build, because a full parts list plus the Thought/Action text doesn't fit in 120 tokens. The **penalty experiment could not be run as planned**: the model rejected `presence_penalty` and `frequency_penalty` (see `dropped_params`), so they were removed automatically.

One limitation of this comparison: the three main Gemini models kept returning rate-limit errors (429), so many runs fell back to `gemini-3.5-flash-lite` (see `models_used`). Some of the differences between setups may therefore come from the model rather than the setting.

**3. How well did the agent reason and use tools?**
In the final run, the agent followed the intended search → price → check order and stayed within budget on every full build (\$1,278, \$1,148 and \$1,278, all under \$1,300 and compatible). Each persona reasoned in its own way:

- **Rex** re-priced the build three times, adjusting parts until it fit.
- **Volt** needed exactly one search, one pricing call and one compatibility check.
- **Maya** checked compatibility *before* pricing, which matches her verify-first persona.

My **first version failed** the full build: the agent searched one category per step (five `search_parts` calls) and hit the 6-step limit before pricing anything. I fixed this by letting `search_parts` take several categories at once (plus a rule telling the agent to do that) and raising the limit to 10 steps. After that, T1 passed for all three personas.

Format errors were common, about 2 per task (Rex 2.0, Maya 2.2, Volt 1.6), usually a missing `Action Input` or extra text. The error-observation loop recovered every time: no run in the final experiments ended at the step limit.

**4. Did the instructions block change the agent's behavior? How?**
Yes, clearly:

- **Removing the worked example (zero-shot) dropped success from 2/2 to 1/2.** It failed the compatibility question, which is exactly what the worked example demonstrates. So the example taught the reasoning pattern for that kind of question, not just the format.
- **Removing the Thought lines (no CoT) also dropped to 1/2.** It failed the multi-step build, even though it had 0 format errors and used about 40% fewer tokens. Thought lines cause most format errors, but they help the agent plan long tasks.
- **The individual rules show up in the traces:**
  - every price in an answer matched a `price_build` observation (rule 2);
  - every full build was compatibility-checked (rule 3) and under budget (rule 4);
  - every policy answer came from `lookup_policy` (rule 5);
  - all personas asked for details on the vague request (rule 6);
  - Volt followed rule 8, a single combined search.

**5. What were the biggest challenges?**

1. **Detecting actions reliably in free text.** I solved this with regular expressions, a stop sequence on `Observation:` so the model can't invent tool results, and error observations that tell the model how to fix its format.
2. **Designing the step budget.** The first version ran out of steps on the full build (see question 3).
3. **Free-tier rate limits.** The main models returned 429 errors throughout. I added a startup check of which models respond, automatic fallback, a 90-second cooldown for rate-limited models, and a resumable experiment log. The side effect is a less clean configuration comparison, because some runs used a different model.
4. **Model-specific parameters.** The model rejected the penalty parameters, so the code detects and logs unsupported settings instead of crashing.

## 8. Chat app (working assistant)

Choose a persona, toggle Chain-of-Thought, and chat. Each reply shows the full **ReAct trace** (thoughts, actions, observations) in a collapsible box above the answer.

In [15]:
import gradio as gr

def respond(message, history, persona, cot):
    past = [{"role": m["role"], "content": m["content"]} for m in history
            if isinstance(m.get("content"), str) and not (m.get("metadata") or {}).get("title")]
    try:
        res = run_agent(message, persona=persona, variant={"cot": cot}, history=past, verbose=False)
    except Exception as e:
        return f"Sorry, the model is busy right now. Please try again in a minute. ({str(e)[:80]})"
    lines = []
    for s in res["trace"]:
        if s["type"] == "llm":
            lines.append("```text\n" + s["text"] + "\n```")
        elif s["type"] == "tool":
            lines.append(f"**👀 Observation** (`{s['tool']}`):\n```text\n{s['observation']}\n```")
        else:
            lines.append(f"⚠️ {s['text']}")
    lines.append(f"*steps: {res['steps']} · tokens: {res['tokens']} · {res['latency']}s*")
    return [{"role": "assistant", "content": "\n\n".join(lines), "metadata": {"title": "🧠 ReAct trace"}},
            {"role": "assistant", "content": res["final"]}]

chat_app = gr.ChatInterface(
    fn=respond, type="messages", title="🛠️ RigSmith PCs: ReAct build advisor",
    description="Pick a persona and plan your PC. Open the 🧠 ReAct trace to see each Thought, Action and Observation.",
    additional_inputs=[gr.Dropdown(list(PERSONAS), value="Rex (Friendly Advisor)", label="Persona"),
                       gr.Checkbox(True, label="Chain-of-Thought (Thought lines)")],
    examples=[[t, "Rex (Friendly Advisor)", True] for t in TASKS.values()],
    theme=gr.themes.Soft(primary_hue="orange"),
)
chat_app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://61c70506ed88052737.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
